# RF-DETR Multi-Class Detection Pipeline: Auto-Discovered Categories
### Model: RF-DETR Base (`resolution=1008`, `optimizer=AdamW`, `lr=1e-4`, `grad_accumulation=8`)
This pipeline fine-tunes an RF-DETR Base model on multi-class annotations with end-to-end diagnostics:
- **Step A (Loss Component Breakdown)**: Tracks classification (CE), bounding box (L1), and GIoU loss components separately to isolate overfitting and query calibration issues.
- **Step B (Label & Category Alignment)**: Automatic pre-flight verification of contiguous 0-indexing, category coverage across splits, class imbalance, and bounding box validity.
- **Step C (Regularization & Full Dataset)**: Full dataset training by default, data augmentations (horizontal flip, color jitter), early stopping patience, and periodic validation mAP monitoring.
- **Effective Batch Size**: `BATCH_SIZE = 2` with `GRAD_ACCUMULATION = 8` (effective batch size = 16).
- **VRAM Monitor**: Real-time tracking of GPU allocated, reserved, peak, and free memory.


In [ ]:
# STEP 0 - Install Dependencies & Clean NumPy Fix
print("=" * 70)
print("[STEP 0] Verifying dependencies & fixing conflicting NumPy files...")
print("=" * 70)

# Purge leftover umath.py from NumPy 2 that breaks NumPy 1.x C-extensions
import os, glob
for pattern in ["/libraries/rfdetr_detection/lib/python*/site-packages/numpy/core/umath.py", "**/numpy/core/umath.py"]:
    for p in glob.glob(pattern, recursive=True):
        try:
            os.remove(p)
            print(f"Removed conflicting leftover file: {p}")
        except Exception as e:
            print(f"Notice on {p}: {e}")

# Cleanly install pinned numpy==1.26.4 and core dependencies
!pip install -q --no-cache-dir "numpy==1.26.4"
!pip install -q --no-cache-dir "torch==2.5.1" "torchvision==0.20.1" --index-url https://download.pytorch.org/whl/cu121
!pip install -q --no-cache-dir "rfdetr==1.4.0"
!pip install -q --no-cache-dir "supervision>=0.22.0"
!pip install -q --no-cache-dir torchmetrics pycocotools pandas opencv-python Pillow matplotlib python-dotenv azure-storage-blob requests tqdm

print("Dependencies verified successfully.")


In [ ]:
# CELL 1 - Imports, VRAM Monitor & Logger Initialization
import os, sys, json, time, math, copy, logging, random, shutil, warnings
warnings.filterwarnings("ignore", message=".*meshgrid.*")
warnings.filterwarnings("ignore", message=".*torch.cuda.amp.autocast.*")
warnings.filterwarnings("ignore", message=".*max_detection_threshold.*")
warnings.filterwarnings("ignore", category=FutureWarning)
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple
from collections import defaultdict, Counter
from urllib.parse import urlparse
from concurrent.futures import ThreadPoolExecutor, as_completed

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(line_buffering=True)
os.environ["PYTHONUNBUFFERED"] = "1"

# Configure multiprocessing sharing strategy to prevent 'Too many open files' error
import torch.multiprocessing as mp
try:
    mp.set_sharing_strategy('file_system')
except Exception:
    pass

try:
    import resource
    rlimit = resource.getrlimit(resource.RLIMIT_NOFILE)
    resource.setrlimit(resource.RLIMIT_NOFILE, (max(rlimit[0], 4096), max(rlimit[1], 4096)))
except Exception:
    pass

import cv2, torch, requests, supervision as sv
# Neutralize Matplotlib interactive callback to permanently prevent _draw_all_if_interactive crash
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.ioff()
plt.close('all')
try:
    _ip = get_ipython()
    if _ip and hasattr(_ip, 'events'):
        _ip.events.callbacks['post_execute'] = [
            _cb for _cb in _ip.events.callbacks.get('post_execute', [])
            if 'draw_all' not in getattr(_cb, '__name__', '')
        ]
except Exception:
    pass
import matplotlib.patches as patches, numpy as np, pandas as pd
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from PIL import Image, ImageDraw, ImageFont
from dotenv import load_dotenv
from tqdm.auto import tqdm

try:
    from torchmetrics.detection.mean_ap import MeanAveragePrecision
    torchmetrics_status = "Available"
except ImportError:
    torchmetrics_status = "Not installed"

try:
    from rfdetr import RFDETRNano, RFDETRSmall, RFDETRMedium, RFDETRBase, RFDETRLarge
    from rfdetr import main as rfdetr_main
    from rfdetr.models.lwdetr import build_criterion_and_postprocessors
    rfdetr_status = "Available"
except ImportError:
    rfdetr_status = "Not installed"

# Auto-flushing console and file logger
class FlushHandler(logging.StreamHandler):
    def emit(self, record):
        super().emit(record)
        self.flush()

class FlushFileHandler(logging.FileHandler):
    def emit(self, record):
        super().emit(record)
        self.flush()

logger = logging.getLogger("rfdetr_multi_class")
logger.setLevel(logging.INFO)
logger.handlers.clear()
logger.propagate = False
log_format = "%(asctime)s | %(levelname)-8s | %(message)s"
console_handler = FlushHandler(sys.stdout)
console_handler.setFormatter(logging.Formatter(log_format))
logger.addHandler(console_handler)

def print_vram_usage(tag="Status"):
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / (1024**3)
        total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        logger.info(f"[VRAM - {tag}] {torch.cuda.get_device_name(0)}: {alloc:.2f} GB / {total:.2f} GB allocated")
    else:
        logger.info(f"[VRAM - {tag}] Running on CPU")

# Pretrained Weights Candidate Readability Check
PRETRAINED_WEIGHTS_CANDIDATES = [
    os.path.expanduser("~/rf-detr-base-coco.pth"),
    "/home/jupyter/rf-detr-base-coco.pth",
    "rf-detr-base-coco.pth"
]

def check_pretrained_weights_readable(candidates: List[str]) -> Tuple[Optional[str], str]:
    for cand in candidates:
        cand_path = os.path.expanduser(cand)
        if os.path.exists(cand_path) and os.access(cand_path, os.R_OK) and os.path.getsize(cand_path) > 1024*1024:
            try:
                ckpt = torch.load(cand_path, map_location="cpu", weights_only=False)
                count = len(ckpt.get("model", ckpt).keys()) if isinstance(ckpt, dict) else "?"
                del ckpt
                return cand_path, f"Readable ({os.path.getsize(cand_path)/1e6:.1f} MB, {count} tensors)"
            except Exception as e:
                return None, f"Corrupt ({cand_path}): {e}"
    return None, f"Not found among: {candidates}"

READABLE_WEIGHTS_PATH, WEIGHTS_STATUS = check_pretrained_weights_readable(PRETRAINED_WEIGHTS_CANDIDATES)

logger.info("=" * 65)
logger.info(f"[CELL 1] Core Libraries & Logger Ready (PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()})")
if READABLE_WEIGHTS_PATH:
    logger.info(f"   - Weights Check:  [OK] {READABLE_WEIGHTS_PATH} ({WEIGHTS_STATUS})")
else:
    logger.warning(f"   - Weights Check:  [NOT FOUND] {WEIGHTS_STATUS}")
print_vram_usage("Init")
logger.info("=" * 65)

In [ ]:
# ==============================================================================
# CELL 2 - MASTER CONFIGURATION & HYPERPARAMETERS
# All pipeline, model, training, evaluation, and inference settings are defined here.
# ==============================================================================

# ------------------------------------------------------------------------------
# 1. Dataset & Pipeline Modes
# ------------------------------------------------------------------------------
PIPELINE_NAME = "multi_class_train_rfdetr"
# Full dataset training by default (SAMPLE_SIZE = None). Set an integer (e.g. 1000) for fast testing.
SAMPLE_SIZE = None
MODE_TAG = f"sample_{SAMPLE_SIZE}" if SAMPLE_SIZE else "full_data"
RANDOM_SEED = 42

# Dataset Split Proportions
TRAIN_RATIO = 0.80
VALID_RATIO = 0.10
TEST_RATIO  = 0.10

# COCO Annotation Field Names
IMAGE_FIELD = "image_id"
CATEGORY_FIELD = "category_id"
BBOX_FIELD = "bbox"

# Azure Storage Download Settings
AZURE_CONNECTION_STRING_ENV = "AZURE_STORAGE_CONNECTION_STRING"
DOWNLOAD_WORKERS = 16

# ------------------------------------------------------------------------------
# 2. Model Architecture
# ------------------------------------------------------------------------------
MODEL_SIZE = "base"      # Options: 'nano', 'small', 'medium', 'base', 'large'
FREEZE_BACKBONE = True   # Freeze DINOv2 ViT backbone: cuts 1 epoch from ~2h to ~30-35m (RECOMMENDED)
PRETRAINED = True
# RF-DETR DINOv2 backbone requires input size divisible by 56 (patch_size 14 * num_windows 4)
# Valid high-res choices: 896 (56*16), 1008 (56*18), 1064 (56*19), 1120 (56*20)
RESOLUTION = 1008        # High-resolution 1008x1008 (divisible by 56, preserves fine 2K details)
if RESOLUTION % 56 != 0:
    RESOLUTION = round(RESOLUTION / 56) * 56
PRETRAINED_WEIGHTS = READABLE_WEIGHTS_PATH if READABLE_WEIGHTS_PATH else "rf-detr-base-coco.pth"

# ------------------------------------------------------------------------------
# 3. Training & Throughput Optimization (Optimized for Fast Epochs on 40k images)
# ------------------------------------------------------------------------------
EPOCHS = 10              # User constraint: max 10 epochs
BATCH_SIZE = 4           # Training batch size (fits safely in 16GB T4 VRAM with frozen backbone)
GRAD_ACCUMULATION = 4    # Effective batch size = BATCH_SIZE * GRAD_ACCUMULATION = 16
STEPS_PER_EPOCH = 2000   # 2,000 steps per epoch (~15 mins/epoch, 20k steps total over 10 epochs)
VAL_BATCH_SIZE = 2       # User constraint: Validation batch size = 2
NUM_WORKERS = 2         # User constraint: max 2 workers
PIN_MEMORY = True        # Pin CUDA memory for zero-copy host-to-device transfers
PERSISTENT_WORKERS = True# Keep DataLoader workers alive between epochs to avoid respawn latency
PREFETCH_FACTOR = 2      # Batches prefetched per worker

# Optimizer & Learning Rate Schedule
LEARNING_RATE = 1e-4     # Base learning rate for AdamW
WEIGHT_DECAY = 1e-4      # L2 regularization
LR_MIN = 1e-6            # Minimum LR for CosineAnnealing schedule
CLIP_GRAD_MAX_NORM = 0.1 # Maximum gradient norm clipping

# ------------------------------------------------------------------------------
# 4. Regularization, Augmentation & Diagnostics (Step A, B, C Checks)
# ------------------------------------------------------------------------------
USE_AUGMENTATION = True        # Step C: Enable Random Horizontal Flip & Color Jitter for train split
EARLY_STOPPING_PATIENCE = 5    # Scaled for 10 epochs
OVERFITTING_RATIO_ALERT = 2.0  # Step A: Trigger alert when Val_Loss / Train_Loss exceeds this ratio
EVAL_MAP_INTERVAL = 1          # Evaluates mAP@50, mAP@75, mAP@50:95, and mAR on every epoch
MAP_EVAL_THRESHOLD = 0.01      # Confidence threshold for mAP (0.01 captures full Precision-Recall curve)

# ------------------------------------------------------------------------------
# 5. Inference, Post-Processing & Visualization
# ------------------------------------------------------------------------------
CONFIDENCE = 0.50              # Minimum confidence score for test detections
CONFIDENCE_THRESHOLD = CONFIDENCE
NMS_THRESHOLD = 0.50           # NMS IoU threshold
NUM_VISUAL_SAMPLES = 4         # Number of ground-truth samples to plot in Cell 13
NUM_INFERENCE_PREVIEWS = 3     # Number of test detection visual plots in Cell 21

# ------------------------------------------------------------------------------
# 6. Directory Structure & File Paths
# ------------------------------------------------------------------------------
REPO_ROOT = Path(".")
PIPELINE_DIR = REPO_ROOT / PIPELINE_NAME
INPUT_JSON_DIR = REPO_ROOT / "coco_files"

DATASET_DIR = os.path.join(PIPELINE_DIR, f"dataset_{MODE_TAG}")
IMAGES_DIR = PIPELINE_DIR / "images"
RAW_IMAGES_DIR = IMAGES_DIR
TRAIN_DIR, VAL_DIR, TEST_DIR = [os.path.join(DATASET_DIR, s) for s in ["train", "val", "test"]]
TRAIN_ANN, VAL_ANN, TEST_ANN = [os.path.join(d, "_annotations.coco.json") for d in [TRAIN_DIR, VAL_DIR, TEST_DIR]]

OUTPUT_DIR = os.path.join(PIPELINE_DIR, f"runs/rf_detr_{MODE_TAG}")
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
LOG_DIR = os.path.join(OUTPUT_DIR, "logs")
INFERENCE_OUTPUT_DIR = os.path.join(PIPELINE_DIR, f"inference_{MODE_TAG}")
FINAL_MODEL_DIR = os.path.join(PIPELINE_DIR, "model")

# ------------------------------------------------------------------------------
# 7. Hardware & Kernel Acceleration Settings
# ------------------------------------------------------------------------------
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    if hasattr(torch, "set_float32_matmul_precision"):
        torch.set_float32_matmul_precision("medium")  # Fast TensorFloat32 math on Ampere/Ada/Hopper

logger.info("=" * 75)
logger.info(f"Master Configuration Loaded: Pipeline={PIPELINE_NAME} ({MODE_TAG})")
logger.info(f"   - Mode:          {'Full Dataset (40k+)' if SAMPLE_SIZE is None else f'Sample Mode ({SAMPLE_SIZE} images)'}")
logger.info(f"   - Batch Size:    Train={BATCH_SIZE} (Eff={BATCH_SIZE * GRAD_ACCUMULATION}), Val={VAL_BATCH_SIZE}, Steps/Epoch={STEPS_PER_EPOCH if STEPS_PER_EPOCH else 'Full'}")
logger.info(f"   - Throughput:    Workers={NUM_WORKERS}, Resolution={RESOLUTION}x{RESOLUTION}, PinMem={PIN_MEMORY}")
logger.info(f"   - Optimization:  LR={LEARNING_RATE}, WeightDecay={WEIGHT_DECAY}, ClipNorm={CLIP_GRAD_MAX_NORM}")
logger.info(f"   - Diagnostics:   FreezeBackbone={FREEZE_BACKBONE}, Aug={USE_AUGMENTATION}, Patience={EARLY_STOPPING_PATIENCE}, mAP_Interval={EVAL_MAP_INTERVAL}")
logger.info("=" * 75)

In [ ]:
# CELL 3 - Dynamic Folder Creation & File Logging
for p in [DATASET_DIR, IMAGES_DIR, OUTPUT_DIR, CHECKPOINT_DIR, LOG_DIR, INFERENCE_OUTPUT_DIR, FINAL_MODEL_DIR]:
    os.makedirs(p, exist_ok=True)

log_file_path = os.path.join(LOG_DIR, "pipeline.log")
for h in [h for h in logger.handlers if isinstance(h, (logging.FileHandler, FlushFileHandler))]:
    logger.removeHandler(h); h.close()

file_handler = FlushFileHandler(log_file_path, mode="a", encoding="utf-8")
file_handler.setFormatter(logging.Formatter(log_format))
logger.addHandler(file_handler)

def tail_log(n: int = 25):
    """View latest log entries from inside Jupyter."""
    if os.path.exists(log_file_path):
        with open(log_file_path, "r", encoding="utf-8") as f:
            print("".join(f.readlines()[-n:]))

load_dotenv()
conn_str = os.getenv(AZURE_CONNECTION_STRING_ENV)
try:
    from azure.storage.blob import BlobServiceClient
    blob_service_client = BlobServiceClient.from_connection_string(conn_str) if conn_str else None
except Exception as e:
    blob_service_client = None

logger.info(f"Directories ready. Active file logger writing to: {log_file_path}")

In [ ]:
# CELL 4 - Read all annotation JSON files from coco_files/
logger.info("=" * 70)
logger.info(f"[CELL 4] Searching for annotation JSON files in: {INPUT_JSON_DIR}")
logger.info("=" * 70)

if not INPUT_JSON_DIR.exists():
    INPUT_JSON_DIR.mkdir(parents=True, exist_ok=True)

json_files = sorted(list(INPUT_JSON_DIR.glob("*.json")))
logger.info(f"   - Found {len(json_files)} JSON file(s) in {INPUT_JSON_DIR}")

raw_records = []
for jf in json_files:
    logger.info(f"   - Reading: {jf.name} ({jf.stat().st_size / 1024:.1f} KB)")
    with open(jf, "r", encoding="utf-8") as f:
        content = f.read().strip()
        if not content:
            continue
        try:
            parsed = json.loads(content)
            if isinstance(parsed, list):
                raw_records.extend(parsed)
            elif isinstance(parsed, dict):
                if "annotations" in parsed and isinstance(parsed["annotations"], list):
                    raw_records.extend(parsed["annotations"])
                else:
                    raw_records.append(parsed)
        except json.JSONDecodeError:
            f.seek(0)
            for line_idx, line in enumerate(f):
                line = line.strip()
                if line:
                    try:
                        parsed_line = json.loads(line)
                        if isinstance(parsed_line, list):
                            raw_records.extend(parsed_line)
                        else:
                            raw_records.append(parsed_line)
                    except Exception as err:
                        logger.warning(f"Error decoding line {line_idx+1} in {jf.name}: {err}")

logger.info(f"Total raw annotation records loaded: {len(raw_records)} from {len(json_files)} file(s).")

In [ ]:
# CELL 5 - Parse Annotations & Auto-Discover Multiple Categories
logger.info("=" * 70)
logger.info("[CELL 5] Parsing annotations and discovering categories dynamically...")
logger.info("=" * 70)

image_annotations: Dict[str, List[Dict[str, Any]]] = {}
category_counts: Dict[str, int] = {}

for item in raw_records:
    if not isinstance(item, dict):
        continue
    img_id = item.get(IMAGE_FIELD)
    if not img_id:
        continue
    
    raw_cat = item.get(CATEGORY_FIELD, ["default"])
    if isinstance(raw_cat, list):
        cat_name = str(raw_cat[0]) if len(raw_cat) > 0 else "default"
    else:
        cat_name = str(raw_cat)
    category_counts[cat_name] = category_counts.get(cat_name, 0) + 1

    raw_bbox = item.get(BBOX_FIELD, [])
    if not (isinstance(raw_bbox, list) and len(raw_bbox) == 4):
        continue
    
    try:
        x, y, w, h = [float(v) for v in raw_bbox]
        if w <= 0 or h <= 0:
            continue
    except (ValueError, TypeError):
        continue

    ann_dict = {
        "bbox": [x, y, w, h],
        "category_name": cat_name,
        "area": float(item.get("area", w * h)),
        "segmentation": item.get("segmentation", [])
    }
    
    if img_id not in image_annotations:
        image_annotations[img_id] = []
    image_annotations[img_id].append(ann_dict)

AUTO_CATEGORIES = sorted(list(category_counts.keys()))
cat_to_id = {cat: idx for idx, cat in enumerate(AUTO_CATEGORIES)}
id_to_cat = {idx: cat for cat, idx in cat_to_id.items()}

for img_id, anns in image_annotations.items():
    for ann in anns:
        ann["category_id"] = cat_to_id[ann["category_name"]]

NUM_CLASSES = len(AUTO_CATEGORIES)
total_boxes = sum(len(v) for v in image_annotations.values())

logger.info(f"   - Unique images found:         {len(image_annotations)}")
logger.info(f"   - Total valid bounding boxes:  {total_boxes}")
logger.info(f"   - Discovered Categories ({NUM_CLASSES}):")
for cat, count in category_counts.items():
    logger.info(f"       - ID {cat_to_id[cat]}: '{cat}' ({count} boxes, {count/total_boxes*100:.1f}%)")
logger.info(f"Discovered {NUM_CLASSES} categories: {AUTO_CATEGORIES}. Total boxes: {total_boxes}")

In [ ]:
# CELL 6 - Group Annotations & Apply Stratified Sampling Mode
logger.info("=" * 70)
logger.info(f"[CELL 6] Applying mode selection: {MODE_TAG.upper()}")
logger.info("=" * 70)

all_image_urls = sorted(list(image_annotations.keys()))
random.seed(RANDOM_SEED)

image_primary_cat: Dict[str, str] = {}
for u in all_image_urls:
    anns = image_annotations[u]
    if anns:
        cats = [a["category_name"] for a in anns]
        image_primary_cat[u] = max(set(cats), key=cats.count)
    else:
        image_primary_cat[u] = AUTO_CATEGORIES[0]

if SAMPLE_SIZE is not None and len(all_image_urls) > SAMPLE_SIZE:
    selected_image_urls = []
    cat_to_images: Dict[str, List[str]] = {cat: [] for cat in AUTO_CATEGORIES}
    for u in all_image_urls:
        cat_to_images[image_primary_cat[u]].append(u)
    
    total_imgs = len(all_image_urls)
    for cat in AUTO_CATEGORIES:
        imgs = cat_to_images[cat]
        target_count = max(1, int(round((len(imgs) / total_imgs) * SAMPLE_SIZE)))
        selected = random.sample(imgs, min(len(imgs), target_count))
        selected_image_urls.extend(selected)
    
    if len(selected_image_urls) > SAMPLE_SIZE:
        selected_image_urls = random.sample(selected_image_urls, SAMPLE_SIZE)
    logger.info(f"   Sample Mode Active: Stratified selection of {len(selected_image_urls)} images.")
else:
    selected_image_urls = all_image_urls
    logger.info(f"   Full Data Mode: Using all {len(selected_image_urls)} images.")

image_urls = sorted(list(set(selected_image_urls)))
filtered_annotations = {u: image_annotations[u] for u in image_urls}
total_selected_boxes = sum(len(v) for v in filtered_annotations.values())

logger.info(f"   - Total images in pipeline:         {len(image_urls)}")
logger.info(f"   - Total bounding boxes in pipeline: {total_selected_boxes}")

In [ ]:
# CELL 7 - Azure URL Helpers & Download Function
logger.info("=" * 70)
logger.info("[CELL 7] Defining Azure Blob download and caching functions...")
logger.info("=" * 70)

IMAGES_DIR = PIPELINE_DIR / "images"

def parse_azure_url(url: str) -> Tuple[Optional[str], Optional[str]]:
    parsed = urlparse(url)
    parts = parsed.path.strip("/").split("/", 1)
    if len(parts) == 2:
        return parts[0], parts[1]
    return None, None

def download_image(url: str, dest_dir: Path) -> Tuple[str, Optional[Path], Optional[str]]:
    filename = Path(urlparse(url).path).name
    dest_path = dest_dir / filename
    if dest_path.exists() and dest_path.stat().st_size > 0:
        return url, dest_path, None

    container_name, blob_name = parse_azure_url(url)
    if blob_service_client and container_name and blob_name:
        try:
            blob_client = blob_service_client.get_blob_client(container=container_name, blob=blob_name)
            with open(dest_path, "wb") as f:
                f.write(blob_client.download_blob().readall())
            return url, dest_path, None
        except Exception as e:
            pass

    try:
        resp = requests.get(url, timeout=30)
        if resp.status_code == 200:
            with open(dest_path, "wb") as f:
                f.write(resp.content)
            return url, dest_path, None
        else:
            return url, None, f"HTTP {resp.status_code}"
    except Exception as e:
        return url, None, str(e)

logger.info("Image download functions ready.")

In [ ]:
# CELL 8 - Parallel Azure Image Downloads
logger.info("=" * 70)
logger.info(f"[CELL 8] Downloading {len(image_urls)} images ({DOWNLOAD_WORKERS} threads)...")
logger.info("=" * 70)

download_results: Dict[str, Dict[str, Any]] = {}
with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as executor:
    futures = {executor.submit(download_image, url, IMAGES_DIR): url for url in image_urls}
    done_count = 0
    for future in as_completed(futures):
        url, path, err = future.result()
        download_results[url] = {"local_path": path, "error": err}
        done_count += 1
        if done_count % 200 == 0 or done_count == len(image_urls):
            logger.info(f"   - Download Progress: {done_count}/{len(image_urls)} processed ({done_count/len(image_urls)*100:.1f}%)...")

success_downloads = [u for u, res in download_results.items() if res["error"] is None]
failed_downloads = [u for u, res in download_results.items() if res["error"] is not None]

logger.info("Download Summary:")
logger.info(f"   - Successfully ready: {len(success_downloads)} images")
logger.info(f"   - Failed:             {len(failed_downloads)} images")

In [ ]:
# CELL 9 - Category-Stratified Train/Val/Test Split (Equal Proportions)
logger.info("=" * 70)
logger.info("[CELL 9] Performing Category-Stratified Split (Train 80%, Val 10%, Test 10%)...")
logger.info("=" * 70)

valid_urls = [u for u in image_urls if download_results.get(u, {}).get("error") is None]
random.seed(RANDOM_SEED)

cat_to_valid_imgs: Dict[str, List[str]] = {cat: [] for cat in AUTO_CATEGORIES}
for u in valid_urls:
    p_cat = image_primary_cat.get(u, AUTO_CATEGORIES[0])
    cat_to_valid_imgs[p_cat].append(u)

train_urls, val_urls, test_urls = [], [], []
for cat, imgs in cat_to_valid_imgs.items():
    random.shuffle(imgs)
    n = len(imgs)
    n_tr = int(n * TRAIN_RATIO)
    n_va = int(n * VALID_RATIO)
    train_urls.extend(imgs[:n_tr])
    val_urls.extend(imgs[n_tr:n_tr + n_va])
    test_urls.extend(imgs[n_tr + n_va:])

def count_category_boxes(urls: List[str]) -> Dict[str, int]:
    counts = {cat: 0 for cat in AUTO_CATEGORIES}
    for u in urls:
        for a in filtered_annotations.get(u, []):
            c = a["category_name"]
            counts[c] = counts.get(c, 0) + 1
    return counts

train_cat_boxes = count_category_boxes(train_urls)
val_cat_boxes = count_category_boxes(val_urls)
test_cat_boxes = count_category_boxes(test_urls)
total_cat_boxes = {cat: train_cat_boxes[cat] + val_cat_boxes[cat] + test_cat_boxes[cat] for cat in AUTO_CATEGORIES}

proportions_rows = []
for cat in AUTO_CATEGORIES:
    tot = max(1, total_cat_boxes[cat])
    proportions_rows.append({
        "Category": cat,
        "Total Boxes": total_cat_boxes[cat],
        "Train Count": train_cat_boxes[cat],
        "Train %": f"{(train_cat_boxes[cat]/tot)*100:.1f}%",
        "Val Count": val_cat_boxes[cat],
        "Val %": f"{(val_cat_boxes[cat]/tot)*100:.1f}%",
        "Test Count": test_cat_boxes[cat],
        "Test %": f"{(test_cat_boxes[cat]/tot)*100:.1f}%",
    })

proportions_df = pd.DataFrame(proportions_rows)
logger.info("Category Proportions Across Splits:")
logger.info("\n" + proportions_df.to_string(index=False))
logger.info(f"Total Images: Train={len(train_urls)}, Val={len(val_urls)}, Test={len(test_urls)}")
logger.info(f"Stratified split verified across {len(AUTO_CATEGORIES)} categories.")

In [ ]:
# CELL 10 - Prepare Split Annotation Directories (Zero-Copy Architecture)
logger.info("=" * 70)
logger.info("[CELL 10] Preparing split annotation directories (Zero-Copy Architecture)...")
logger.info("=" * 70)

for split_name in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    os.makedirs(split_name, exist_ok=True)
    images_link = os.path.join(split_name, "images")
    if not os.path.exists(images_link):
        try:
            os.symlink(os.path.abspath(IMAGES_DIR), images_link)
        except OSError:
            pass

logger.info(f"   - Shared images directory:  {IMAGES_DIR} ({len(os.listdir(IMAGES_DIR))} files)")
logger.info(f"   - Train split annotations:  {TRAIN_DIR} ({len(train_urls)} images)")
logger.info(f"   - Val split annotations:    {VAL_DIR} ({len(val_urls)} images)")
logger.info(f"   - Test split annotations:   {TEST_DIR} ({len(test_urls)} images)")
logger.info("Split directories ready with zero redundant image copying.")

In [ ]:
# CELL 11 - Build Multi-Class COCO Annotations
logger.info("=" * 70)
logger.info(f"[CELL 11] Building COCO annotations for {NUM_CLASSES} discovered categories...")
logger.info("=" * 70)

categories_def = [{"id": idx, "name": cat, "supercategory": "none"} for cat, idx in cat_to_id.items()]

def build_coco_for_split(urls: List[str], split_dir: str, ann_path: str) -> Dict[str, Any]:
    images_list = []
    annotations_list = []
    ann_id = 1

    for img_id_idx, url in enumerate(urls, start=1):
        local_path = download_results[url]["local_path"]
        if not (local_path and local_path.exists()):
            continue
        try:
            with Image.open(local_path) as im:
                width, height = im.size
        except Exception:
            width, height = 1920, 1080

        filename = local_path.name
        images_list.append({
            "id": img_id_idx,
            "file_name": filename,
            "width": int(width),
            "height": int(height),
            "original_url": url
        })

        for ann in filtered_annotations.get(url, []):
            x, y, w, h = ann["bbox"]
            cid = ann["category_id"]
            annotations_list.append({
                "id": ann_id,
                "image_id": img_id_idx,
                "category_id": int(cid),
                "bbox": [round(x, 2), round(y, 2), round(w, 2), round(h, 2)],
                "area": round(w * h, 2),
                "iscrowd": 0,
                "segmentation": []
            })
            ann_id += 1

    coco_dict = {
        "info": {"description": f"RF-DETR Multi-Class Dataset ({NUM_CLASSES} classes)", "version": "1.0"},
        "licenses": [],
        "images": images_list,
        "annotations": annotations_list,
        "categories": categories_def
    }
    
    with open(ann_path, "w", encoding="utf-8") as f:
        json.dump(coco_dict, f, indent=2)
    logger.info(f"   - Saved: {ann_path} ({len(images_list)} images, {len(annotations_list)} boxes)")
    return coco_dict

build_coco_for_split(train_urls, TRAIN_DIR, TRAIN_ANN)
build_coco_for_split(val_urls, VAL_DIR, VAL_ANN)
build_coco_for_split(test_urls, TEST_DIR, TEST_ANN)

logger.info("Multi-Class COCO JSON files created.")

In [ ]:
# CELL 12 - Dataset Inspection & Step B Label/Category Alignment Verification
def load_coco_info(annotation_path: str) -> dict:
    """Parse a COCO annotation file and return summary statistics."""
    with open(annotation_path) as f:
        data = json.load(f)
    categories = {c["id"]: c["name"] for c in data.get("categories", [])}
    num_images = len(data.get("images", []))
    num_anns = len(data.get("annotations", []))
    ann_per_cat: dict = {}
    for ann in data.get("annotations", []):
        cat_name = categories.get(ann["category_id"], "unknown")
        ann_per_cat[cat_name] = ann_per_cat.get(cat_name, 0) + 1
    return {
        "num_images": num_images,
        "num_anns": num_anns,
        "num_classes": len(categories),
        "categories": categories,
        "ann_per_cat": ann_per_cat,
        "raw": data,
    }

# Validate all annotation files exist
for split, path in [("train", TRAIN_ANN), ("val", VAL_ANN), ("test", TEST_ANN)]:
    assert os.path.exists(path), f"Missing annotation file for '{split}': {path}"
logger.info("All annotation files verified.")

train_info = load_coco_info(TRAIN_ANN)
val_info = load_coco_info(VAL_ANN)
test_info = load_coco_info(TEST_ANN)

# NUM_CLASSES is derived from the training annotation file
NUM_CLASSES = train_info["num_classes"]

logger.info("=" * 58)
logger.info(f"{'Split': <10} {'Images': >8} {'Annotations': >14} {'Classes': >9}")
logger.info("-" * 58)
for name, info in [("train", train_info), ("val", val_info), ("test", test_info)]:
    logger.info(f"{name: <10} {info['num_images']: >8} {info['num_anns']: >14} {info['num_classes']: >9}")
logger.info("=" * 58)
logger.info(f"Classes ({NUM_CLASSES} total): {train_info['categories']}")

# ==========================================================================
# STEP B CHECKS: Comprehensive Label & Category Alignment Diagnostic Suite
# ==========================================================================
from collections import defaultdict, Counter
def verify_label_and_category_alignment(train_path: str, val_path: str, test_path: str) -> dict:
    """Rigorous Step B diagnostic check verifying category indexing, split coverage, and box coordinates."""
    logger.info("=" * 75)
    logger.info("[STEP B CHECK] Executing Label & Category Alignment Verification Suite...")
    logger.info("=" * 75)
    
    issues = []
    splits = {"train": train_path, "val": val_path, "test": test_path}
    loaded = {}
    for sname, spath in splits.items():
        with open(spath, "r", encoding="utf-8") as f:
            loaded[sname] = json.load(f)

    # Check 1: Contiguous 0-indexing in categories
    train_cats = loaded["train"].get("categories", [])
    train_ids = sorted([c["id"] for c in train_cats])
    expected_ids = list(range(len(train_cats)))
    if train_ids != expected_ids:
        msg = f"Category IDs are not contiguous 0-indexed! Expected {expected_ids[:5]}..., got {train_ids[:5]}..."
        issues.append(msg)
        logger.error(f"   [FAIL] {msg}")
    else:
        logger.info(f"   [PASS] Contiguous 0-indexing verified: IDs {min(train_ids)} to {max(train_ids)} ({len(train_ids)} classes).")

    # Check 2: Cross-split category consistency
    train_cat_dict = {c["id"]: c["name"] for c in train_cats}
    for sname in ["val", "test"]:
        s_ids = set(c["id"] for c in loaded[sname].get("categories", []))
        diff = s_ids - set(train_ids)
        if diff:
            msg = f"{sname.capitalize()} contains category IDs not present in Train: {diff}"
            issues.append(msg)
            logger.error(f"   [FAIL] {msg}")
        else:
            logger.info(f"   [PASS] All {sname} categories align exactly with train categories.")

    # Check 3: Bounding box validity and class counts
    ann_counts = {s: defaultdict(int) for s in splits}
    degenerate_boxes = 0
    for sname, data in loaded.items():
        imgs = {im["id"]: im for im in data.get("images", [])}
        for ann in data.get("annotations", []):
            cid = ann.get("category_id")
            ann_counts[sname][cid] += 1
            bbox = ann.get("bbox", [])
            if len(bbox) != 4 or bbox[2] <= 0 or bbox[3] <= 0:
                degenerate_boxes += 1
                continue
            x, y, w, h = bbox
            im_meta = imgs.get(ann.get("image_id"), {})
            im_w, im_h = im_meta.get("width", 1920), im_meta.get("height", 1080)
            if x < 0 or y < 0 or (x + w) > im_w * 1.05 or (y + h) > im_h * 1.05:
                degenerate_boxes += 1

    logger.info("-" * 75)
    logger.info(f"{'ID':<5} {'Category Name':<25} {'Train Boxes':>12} {'Val Boxes':>11} {'Test Boxes':>11} {'Status':>8}")
    logger.info("-" * 75)
    for cid in train_ids:
        cname = train_cat_dict.get(cid, "Unknown")
        tr_c = ann_counts["train"][cid]
        va_c = ann_counts["val"][cid]
        te_c = ann_counts["test"][cid]
        status = "OK"
        if va_c == 0:
            status = "NO VAL!"
            issues.append(f"Category '{cname}' (ID {cid}) has 0 instances in validation split!")
        elif tr_c < 5:
            status = "LOW DATA"
            issues.append(f"Category '{cname}' (ID {cid}) has only {tr_c} train instances (<5).")
        logger.info(f"{cid:<5} {cname[:24]:<25} {tr_c:>12} {va_c:>11} {te_c:>11} {status:>8}")
    logger.info("-" * 75)

    if degenerate_boxes > 0:
        msg = f"Detected {degenerate_boxes} degenerate or out-of-bounds bounding boxes across dataset splits!"
        issues.append(msg)
        logger.warning(f"   [WARN] {msg}")
    else:
        logger.info("   [PASS] All bounding boxes verified: valid positive width/height within image limits.")

    if not issues:
        logger.info("[STEP B CHECK PASSED] Dataset labels, indexing, and splits are fully consistent.")
    else:
        logger.warning(f"[STEP B CHECK NOTICE] {len(issues)} item(s) to be aware of. Validation loss may be affected by class distribution.")
    logger.info("=" * 75)
    return {"issues": issues, "ann_counts": ann_counts}

step_b_audit = verify_label_and_category_alignment(TRAIN_ANN, VAL_ANN, TEST_ANN)

In [ ]:
# CELL 13 - Print Sample Images per Tag (In-Memory Display - No Files Saved)
from PIL import Image, ImageDraw
from IPython.display import display
from collections import defaultdict

# High-contrast color palette for distinct multi-class boxes
SAMPLE_COLORS = [
    (230, 25, 75),   # Red
    (60, 180, 75),   # Green
    (255, 225, 25),  # Yellow
    (67, 99, 216),   # Blue
    (245, 130, 49),  # Orange
    (145, 30, 180),  # Purple
    (66, 212, 244),  # Cyan
    (240, 50, 230),  # Magenta
    (191, 239, 69),  # Lime
    (250, 190, 212), # Pink
    (70, 153, 144),  # Teal
    (220, 190, 255)  # Lavender
]

def print_sample_images_per_tag(annotation_path: str, image_dir: str, max_display_width: int = 700):
    """
    Selects sample images covering each discovered category tag, annotates
    bounding boxes with colored badges in memory, and prints/displays them
    inline directly in the notebook without saving any files to disk.
    """
    print("=" * 75)
    print("[CELL 13] Printing Sample Images with Category Tags (No Disk Saving)")
    print("=" * 75)

    with open(annotation_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    categories = {c["id"]: c["name"] for c in data.get("categories", [])}
    img_id_map = {im["id"]: im for im in data.get("images", [])}
    
    cat_to_images = defaultdict(list)
    img_to_anns = defaultdict(list)
    for ann in data.get("annotations", []):
        cid = ann.get("category_id")
        iid = ann.get("image_id")
        cat_to_images[cid].append(iid)
        img_to_anns[iid].append(ann)

    # Match with existing local image files
    existing_cat_images = {}
    for cid, iids in cat_to_images.items():
        valid_iids = [
            iid for iid in iids
            if iid in img_id_map and os.path.exists(os.path.join(image_dir, img_id_map[iid]["file_name"]))
        ]
        if valid_iids:
            existing_cat_images[cid] = valid_iids

    if not existing_cat_images:
        print("[WARN] No valid local images found to display.")
        return

    # Display an annotated sample for every distinct category tag
    for cid in sorted(existing_cat_images.keys()):
        cat_name = categories.get(cid, str(cid))
        sample_img_id = existing_cat_images[cid][0]
        meta = img_id_map[sample_img_id]
        fname = meta["file_name"]
        img_path = os.path.join(image_dir, fname)

        img = Image.open(img_path).convert("RGB")
        im_w, im_h = img.size
        draw = ImageDraw.Draw(img)
        anns = img_to_anns.get(sample_img_id, [])

        cat_counts = defaultdict(int)
        for ann in anns:
            raw_bbox = ann.get("bbox", [])
            if len(raw_bbox) != 4:
                continue
            x, y, w, h = [float(v) for v in raw_bbox]
            ann_cid = int(ann.get("category_id", 0))
            ann_cname = str(categories.get(ann_cid, ann_cid))
            cat_counts[ann_cname] += 1
            color = SAMPLE_COLORS[ann_cid % len(SAMPLE_COLORS)]

            # Clamp box to valid dimensions
            x0 = max(0.0, min(float(im_w - 1), x))
            y0 = max(0.0, min(float(im_h - 1), y))
            x1 = max(0.0, min(float(im_w - 1), x + w))
            y1 = max(0.0, min(float(im_h - 1), y + h))

            # Draw bounding box
            draw.rectangle([x0, y0, x1, y1], outline=color, width=4)

            # Draw colored label badge with tag name
            label_text = f" {ann_cname} "
            if hasattr(draw, "textbbox"):
                tb = draw.textbbox((x0, max(0.0, y0 - 18.0)), label_text)
                draw.rectangle(tb, fill=color)
                draw.text((x0, max(0.0, y0 - 18.0)), label_text, fill=(255, 255, 255))
            else:
                draw.text((x0 + 2, max(0.0, y0 - 14.0)), label_text, fill=color)

        summary_str = ", ".join(f"'{k}': {v}" for k, v in sorted(cat_counts.items()))
        print(f"\n>>> Target Tag: [{cat_name}] | File: {fname} ({im_w}x{im_h}) | Boxes: {len(anns)} ({summary_str})")

        # Scale in-memory copy for clean notebook view if wide
        if im_w > max_display_width:
            scale = max_display_width / im_w
            display_img = img.resize((max_display_width, int(im_h * scale)), Image.BILINEAR)
        else:
            display_img = img

        # Directly print/display in notebook (Zero files saved to disk)
        display(display_img)

    print("\n" + "=" * 75)
    print("All category tag samples displayed inline successfully.")
    print("=" * 75)

# Aliases for compatibility with earlier calls
visualize_samples = lambda *args, **kwargs: print_sample_images_per_tag(TRAIN_ANN, str(IMAGES_DIR))
print_and_visualize_samples = visualize_samples

print_sample_images_per_tag(TRAIN_ANN, str(IMAGES_DIR))


In [ ]:
# CELL 14 - Safe Pretrained Weights Loading & Detection Head Patch
from rfdetr.models.lwdetr import LWDETR

# 1. Patch LWDETR.load_state_dict using torch.nn.Module.load_state_dict directly (prevents RecursionError)
def _safe_lwdetr_load_state_dict(self, state_dict, strict=True):
    """Filters out classification heads (class_embed, enc_out_class_embed) from checkpoint when num_classes differs."""
    model_state = self.state_dict()
    filtered_state_dict = {}
    mismatched = []
    
    for k, v in state_dict.items():
        clean_k = k
        if clean_k not in model_state and clean_k.startswith("model."):
            clean_k = clean_k[6:]
        if clean_k not in model_state and clean_k.startswith("module."):
            clean_k = clean_k[7:]
            
        if clean_k in model_state:
            if model_state[clean_k].shape == v.shape:
                filtered_state_dict[clean_k] = v
            else:
                mismatched.append((clean_k, tuple(v.shape), tuple(model_state[clean_k].shape)))
        elif k in model_state:
            if model_state[k].shape == v.shape:
                filtered_state_dict[k] = v
            else:
                mismatched.append((k, tuple(v.shape), tuple(model_state[k].shape)))

    if mismatched:
        logger.info(f"Notice: Filtered {len(mismatched)} classification head layers with class-count mismatch from COCO weights:")
        for name, ckpt_shape, model_shape in mismatched[:4]:
            logger.info(f"   - {name}: checkpoint {ckpt_shape} -> target model {model_shape}")
        if len(mismatched) > 4:
            logger.info(f"   - ... and {len(mismatched) - 4} more classification head tensors.")
        logger.info("Pretrained backbone, transformer, and regression heads loaded successfully.")
    
    # Call base PyTorch nn.Module.load_state_dict directly to avoid any recursion on re-runs
    return torch.nn.Module.load_state_dict(self, filtered_state_dict, strict=False)

LWDETR.load_state_dict = _safe_lwdetr_load_state_dict
logger.info("LWDETR.load_state_dict patched for safe custom num_classes loading.")

# 2. Patch reinitialize_detection_head to handle both class_embed and transformer.enc_out_class_embed
def _fixed_reinitialize(self, num_classes):
    lwdetr = self.model  # the actual LWDETR nn.Module
    device = next(lwdetr.parameters()).device
    
    # Reinitialize class_embed
    if hasattr(lwdetr, "class_embed"):
        in_feat = lwdetr.class_embed.in_features
        lwdetr.class_embed = nn.Linear(in_feat, num_classes).to(device)
        nn.init.normal_(lwdetr.class_embed.weight, std=0.01)
        nn.init.zeros_(lwdetr.class_embed.bias)
        
    # Reinitialize transformer.enc_out_class_embed
    if hasattr(lwdetr, "transformer") and hasattr(lwdetr.transformer, "enc_out_class_embed"):
        enc_heads = lwdetr.transformer.enc_out_class_embed
        if isinstance(enc_heads, nn.ModuleList):
            for i in range(len(enc_heads)):
                in_feat = enc_heads[i].in_features
                enc_heads[i] = nn.Linear(in_feat, num_classes).to(device)
                nn.init.normal_(enc_heads[i].weight, std=0.01)
                nn.init.zeros_(enc_heads[i].bias)
        elif isinstance(enc_heads, nn.Linear):
            in_feat = enc_heads.in_features
            lwdetr.transformer.enc_out_class_embed = nn.Linear(in_feat, num_classes).to(device)
            nn.init.normal_(lwdetr.transformer.enc_out_class_embed.weight, std=0.01)
            nn.init.zeros_(lwdetr.transformer.enc_out_class_embed.bias)
            
    logger.info(f"Detection heads (class_embed + enc_out_class_embed) initialized for num_classes={num_classes}")

rfdetr_main.Model.reinitialize_detection_head = _fixed_reinitialize
logger.info("reinitialize_detection_head patched.")

# 3. Model Instantiation
model_cls_map = {
    "nano": RFDETRNano,
    "small": RFDETRSmall,
    "medium": RFDETRMedium,
    "base": RFDETRBase,
    "large": RFDETRLarge
}
ModelClass = model_cls_map.get(MODEL_SIZE.lower(), RFDETRBase)
model = ModelClass(
    num_classes=NUM_CLASSES,
    pretrain_weights=PRETRAINED_WEIGHTS,
    resolution=RESOLUTION
)
logger.info(f"RF-DETR-{MODEL_SIZE.capitalize()} instantiated with resolution={RESOLUTION}, num_classes={NUM_CLASSES}")

# Explicitly ensure detection heads match NUM_CLASSES
if hasattr(model, "model") and hasattr(model.model, "reinitialize_detection_head"):
    model.model.reinitialize_detection_head(NUM_CLASSES)

# Build ordered list of class names from COCO categories
class_names = [
    train_info["categories"][cid]
    for cid in sorted(train_info["categories"].keys())
]
logger.info(f"class_names: {class_names}")

In [ ]:
# CELL 15 - Optimized PyTorch Dataset with Training Augmentations (Fast OpenCV C++ Decoding)
class COCODetectionDataset(Dataset):
    """Fast COCO dataset using OpenCV with optional data augmentations for training regularization."""
    def __init__(self, img_dir: str, ann_file: str, resolution: int = 560, is_train: bool = False, use_augmentation: bool = False):
        with open(ann_file) as f:
            data = json.load(f)
        self.img_dir = img_dir
        self.resolution = resolution
        self.is_train = is_train
        self.use_augmentation = use_augmentation and is_train
        self.cat_to_id = {c: i for i, c in enumerate(sorted(c["id"] for c in data["categories"]))}
        ann_by_img: dict = {}
        for ann in data["annotations"]:
            ann_by_img.setdefault(ann["image_id"], []).append(ann)
        self.samples = [(img, ann_by_img[img["id"]]) for img in data["images"] if img["id"] in ann_by_img]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_meta, anns = self.samples[idx]
        img_path = os.path.join(self.img_dir, img_meta["file_name"])

        # High-performance OpenCV decode & resize (3x faster than PIL on CPU)
        cv_img = cv2.imread(img_path)
        if cv_img is not None:
            orig_h, orig_w = cv_img.shape[:2]
            # Step C Augmentation: Random Horizontal Flip during training
            do_hflip = self.use_augmentation and (random.random() < 0.5)
            if do_hflip:
                cv_img = cv2.flip(cv_img, 1)

            # Step C Augmentation: Mild Color/Brightness jitter
            if self.use_augmentation and (random.random() < 0.5):
                alpha = 1.0 + random.uniform(-0.15, 0.15)  # Contrast
                beta = random.uniform(-15, 15)             # Brightness
                cv_img = np.clip(alpha * cv_img + beta, 0, 255).astype(np.uint8)

            resized = cv2.resize(cv_img, (self.resolution, self.resolution), interpolation=cv2.INTER_LINEAR)
            rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)
            img_t = torch.from_numpy(rgb).permute(2, 0, 1).float().div_(255.0)
        else:
            with Image.open(img_path).convert("RGB") as pil_im:
                orig_w, orig_h = pil_im.size
                do_hflip = self.use_augmentation and (random.random() < 0.5)
                if do_hflip:
                    pil_im = TF.hflip(pil_im)
                resized = pil_im.resize((self.resolution, self.resolution), Image.BILINEAR)
                img_t = TF.to_tensor(resized)

        img_t = TF.normalize(img_t, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

        boxes, labels = [], []
        for ann in anns:
            x, y, w, h = ann["bbox"]
            if w <= 0 or h <= 0: continue
            
            cx = (x + w / 2) / orig_w
            cy = (y + h / 2) / orig_h
            bw = w / orig_w
            bh = h / orig_h
            if do_hflip:
                cx = 1.0 - cx

            boxes.append([
                float(np.clip(cx, 0, 1)),
                float(np.clip(cy, 0, 1)),
                float(np.clip(bw, 0, 1)),
                float(np.clip(bh, 0, 1))
            ])
            labels.append(self.cat_to_id[ann["category_id"]])

        return img_t, {
            "boxes": torch.tensor(boxes, dtype=torch.float32) if boxes else torch.zeros((0, 4)),
            "labels": torch.tensor(labels, dtype=torch.long) if labels else torch.zeros(0, dtype=torch.long),
            "image_id": torch.tensor([img_meta["id"]]),
            "orig_size": torch.tensor([orig_h, orig_w]),
            "size": torch.tensor([self.resolution, self.resolution]),
        }

def collate_fn(batch):
    return torch.stack([item[0] for item in batch]), [item[1] for item in batch]

logger.info(f"Fast COCODetectionDataset ready with data augmentation support (is_train={USE_AUGMENTATION}).")

In [ ]:
# CELL 16 - Criterion, Asynchronous DataLoaders, AdamW & Step C Early Stopping Checkpointer
args = copy.deepcopy(model.model.args)
args.num_classes = NUM_CLASSES
args.device = "cuda" if torch.cuda.is_available() else "cpu"
criterion, _ = build_criterion_and_postprocessors(args)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lwdetr = model.model.model
lwdetr.to(device)
criterion.to(device)

# Step C: Instantiate datasets using master settings
train_ds = COCODetectionDataset(str(IMAGES_DIR), TRAIN_ANN, RESOLUTION, is_train=True, use_augmentation=USE_AUGMENTATION)
val_ds = COCODetectionDataset(str(IMAGES_DIR), VAL_ANN, RESOLUTION, is_train=False, use_augmentation=False)

# High-throughput asynchronous DataLoaders configured from Master Settings
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    drop_last=True,
    pin_memory=PIN_MEMORY,
    persistent_workers=(NUM_WORKERS > 0 and PERSISTENT_WORKERS),
    prefetch_factor=PREFETCH_FACTOR if NUM_WORKERS > 0 else None
)
val_loader = DataLoader(
    val_ds,
    batch_size=VAL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=PIN_MEMORY,
    persistent_workers=(NUM_WORKERS > 0 and PERSISTENT_WORKERS),
    prefetch_factor=PREFETCH_FACTOR if NUM_WORKERS > 0 else None
)

# Backbone Freezing Control for Massive Speedup
if FREEZE_BACKBONE and hasattr(lwdetr, "backbone"):
    for p in lwdetr.backbone.parameters():
        p.requires_grad = False
    logger.info("DINOv2 backbone frozen: training only DETR encoder/decoder & heads (~60% speedup).")
else:
    logger.info("Full model fine-tuning enabled (including DINOv2 backbone).")

trainable_params = [p for p in lwdetr.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR_MIN)

# Step C: Enhanced Checkpointer with Early Stopping and Full History Logging
class BestValLossCheckpointSD:
    def __init__(self, checkpoint_dir: str, module: nn.Module, patience: int = EARLY_STOPPING_PATIENCE, verbose: bool = True):
        self.ckpt_dir = Path(checkpoint_dir); self.ckpt_dir.mkdir(parents=True, exist_ok=True)
        self.module, self.verbose = module, verbose
        self.patience = patience
        self.best_loss, self.best_epoch, self.history = float("inf"), -1, []
        self.consecutive_no_improve = 0
        self.should_stop = False
        self.log_path, self.best_path = self.ckpt_dir / "training_log.json", self.ckpt_dir / "best_model.pth"

    def step(self, epoch: int, train_loss: float, val_loss: float, train_comp: dict = None, val_comp: dict = None, val_map50: float = None, val_map: float = None) -> bool:
        improved = val_loss < self.best_loss
        if improved:
            self.best_loss, self.best_epoch = val_loss, epoch
            self.consecutive_no_improve = 0
            torch.save(self.module.state_dict(), str(self.best_path))
            torch.save(self.module.state_dict(), str(self.ckpt_dir / f"epoch_{epoch:04d}_val{val_loss:.4f}.pth"))
        else:
            self.consecutive_no_improve += 1
            if self.consecutive_no_improve >= self.patience:
                self.should_stop = True

        ratio = round(val_loss / max(train_loss, 1e-6), 2)
        entry = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_train_ratio": ratio,
            "train_components": train_comp or {},
            "val_components": val_comp or {},
            "val_map50": val_map50,
            "val_map50_95": val_map,
            "time": time.strftime("%Y-%m-%dT%H:%M:%S")
        }
        self.history.append(entry)
        with open(self.log_path, "w") as f:
            json.dump({"best_epoch": self.best_epoch, "best_val_loss": self.best_loss, "patience": self.patience, "history": self.history}, f, indent=2)

        return improved

    def load_best(self):
        self.module.load_state_dict(torch.load(str(self.best_path), map_location="cpu"))
        logger.info(f"Loaded best checkpoint (Epoch {self.best_epoch}, Val Loss: {self.best_loss:.4f})")

ckpt = BestValLossCheckpointSD(CHECKPOINT_DIR, lwdetr, patience=EARLY_STOPPING_PATIENCE)
logger.info(f"DataLoaders: Train batches={len(train_loader)} (B={BATCH_SIZE}), Val batches={len(val_loader)} (B={VAL_BATCH_SIZE})")
print_vram_usage("Pre-Training")


In [ ]:
# CELL 17 - Optimized Training Step (AMP, Accumulation) & Step A Loss Breakdown Tracking
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

from collections import defaultdict
def run_epoch(model, loader, criterion, optimizer, device, is_train: bool, epoch: int, num_epochs: int, max_steps: Optional[int] = None):
    """Runs one training or validation epoch cleanly without noisy step-by-step logs."""
    model.train(is_train)
    phase = "train" if is_train else "val"
    total, count = 0.0, 0
    component_totals = defaultdict(float)
    total_steps = min(len(loader), max_steps) if (is_train and max_steps) else len(loader)
    pbar = tqdm(loader, total=total_steps, desc=f"Epoch {epoch:>2}/{num_epochs} [{phase}]", leave=False, unit="batch", dynamic_ncols=True)

    with torch.set_grad_enabled(is_train):
        for step, (images, targets) in enumerate(pbar):
            if is_train and max_steps and (step >= max_steps):
                break
            images = images.to(device, non_blocking=True)
            targets = [{k: v.to(device, non_blocking=True) for k, v in t.items()} for t in targets]

            with torch.amp.autocast(device_type="cuda", enabled=torch.cuda.is_available()):
                outputs = model(images)
                loss_dict = criterion(outputs, targets)
                loss = sum(loss_dict[k] * criterion.weight_dict[k] for k in loss_dict if k in criterion.weight_dict)

            if is_train:
                loss_scaled = loss / GRAD_ACCUMULATION
                scaler.scale(loss_scaled).backward()
                if (step + 1) % GRAD_ACCUMULATION == 0 or (step + 1) == len(loader):
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), max_norm=CLIP_GRAD_MAX_NORM)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)

            # Silently track components for log JSON
            for k, v in loss_dict.items():
                if k in criterion.weight_dict:
                    val_w = (v * criterion.weight_dict[k]).item()
                    if "loss_ce" in k:
                        component_totals["loss_ce"] += val_w
                    elif "loss_bbox" in k:
                        component_totals["loss_bbox"] += val_w
                    elif "loss_giou" in k:
                        component_totals["loss_giou"] += val_w
                    else:
                        component_totals[k] += val_w

            total += loss.item()
            count += 1
            pbar.set_postfix({"loss": f"{(total / count):.4f}"})

    pbar.close()
    if torch.cuda.is_available() and not is_train:
        torch.cuda.empty_cache()

    epoch_avg = total / max(count, 1)
    comp_avgs = {k: round(v / max(count, 1), 4) for k, v in component_totals.items()}
    return epoch_avg, comp_avgs

def box_cxcywh_to_xyxy(boxes: torch.Tensor) -> torch.Tensor:
    cx, cy, w, h = boxes.unbind(-1)
    return torch.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], dim=-1)

def compute_map(model, loader, device, img_size: int, threshold: float = MAP_EVAL_THRESHOLD) -> dict:
    model.eval()
    metric = MeanAveragePrecision(iou_type="bbox", class_metrics=True, max_detection_thresholds=[1, 10, 300])
    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device, non_blocking=True)
            with torch.amp.autocast(device_type="cuda", enabled=torch.cuda.is_available()):
                outputs = model(images)
            pred_logits, pred_boxes = outputs["pred_logits"], outputs["pred_boxes"]
            preds_list, tgts_list = [], []
            for i in range(len(images)):
                scores, lbs = pred_logits[i].sigmoid().max(-1)
                keep = scores > threshold
                preds_list.append({
                    "boxes": (box_cxcywh_to_xyxy(pred_boxes[i][keep]) * img_size).cpu().float(),
                    "scores": scores[keep].cpu().float(),
                    "labels": lbs[keep].cpu(),
                })
                tgts_list.append({
                    "boxes": (box_cxcywh_to_xyxy(targets[i]["boxes"]) * img_size).cpu().float(),
                    "labels": targets[i]["labels"].cpu(),
                })
            metric.update(preds_list, tgts_list)
    return metric.compute()

logger.info("Compiled optimized training and compute_map steps with Step A loss breakdown.")


In [ ]:
# CELL 18 - Training Loop with Loss Breakdown, mAP@50, mAP@75, mAP@50:95 & mAR
print("=" * 95)
print(f"Training Started: {EPOCHS} Epochs | Batch: {BATCH_SIZE} (Effective: {BATCH_SIZE*GRAD_ACCUMULATION}) | Res: {RESOLUTION}x{RESOLUTION}")
print("=" * 95)

for epoch in range(1, EPOCHS + 1):
    t_start = time.time()
    tr_loss, tr_comp = run_epoch(lwdetr, train_loader, criterion, optimizer, device, is_train=True, epoch=epoch, num_epochs=EPOCHS, max_steps=STEPS_PER_EPOCH)
    val_loss, val_comp = run_epoch(lwdetr, val_loader, criterion, optimizer, device, is_train=False, epoch=epoch, num_epochs=EPOCHS)
    
    current_lr = optimizer.param_groups[0]["lr"]
    scheduler.step()

    # Validation detection metrics: mAP@50, mAP@75, mAP@50:95, and mAR@100
    val_map50, val_map75, val_map, val_mar = None, None, None, None
    metrics_str = "mAP@50: - | mAP@75: - | mAP@50:95: - | mAR: -"
    if epoch % EVAL_MAP_INTERVAL == 0:
        map_metrics = compute_map(lwdetr, val_loader, device, img_size=RESOLUTION, threshold=MAP_EVAL_THRESHOLD)
        def _to_float(key):
            val = map_metrics.get(key, 0.0)
            return float(val.item()) if hasattr(val, "item") else float(val)
        val_map50 = _to_float("map_50")
        val_map75 = _to_float("map_75")
        val_map   = _to_float("map")
        val_mar   = _to_float("mar_100")
        metrics_str = f"mAP@50: {val_map50:.4f} | mAP@75: {val_map75:.4f} | mAP@50:95: {val_map:.4f} | mAR: {val_mar:.4f}"

    improved = ckpt.step(epoch, tr_loss, val_loss, train_comp=tr_comp, val_comp=val_comp, val_map50=val_map50, val_map=val_map)
    tag = " <-- [BEST MODEL SAVED]" if improved else ""
    elapsed = time.time() - t_start
    time_str = f"{int(elapsed // 60)}m {int(elapsed % 60):02d}s"

    tr_sub = f"cls={tr_comp.get('loss_ce', 0):.2f}, box={tr_comp.get('loss_bbox', 0):.2f}, giou={tr_comp.get('loss_giou', 0):.2f}"
    val_sub = f"cls={val_comp.get('loss_ce', 0):.2f}, box={val_comp.get('loss_bbox', 0):.2f}, giou={val_comp.get('loss_giou', 0):.2f}"

    # Clean, comprehensive 3-line status per epoch
    print(f"\nEpoch {epoch:02d}/{EPOCHS} [{time_str}] | LR: {current_lr:.1e} | Best Val: {ckpt.best_loss:.4f}{tag}")
    print(f"  • Loss   -> Train: {tr_loss:.4f} ({tr_sub}) | Val: {val_loss:.4f} ({val_sub})")
    print(f"  • Metrics-> {metrics_str}")

    # Overfitting check: only print if gap is significant
    gap_ratio = val_loss / max(tr_loss, 1e-6)
    if gap_ratio >= OVERFITTING_RATIO_ALERT and epoch >= 3:
        print(f"  [Notice] High Val/Train loss ratio ({gap_ratio:.1f}x) - model may be overfitting.")

    if ckpt.should_stop:
        print("\n" + "=" * 95)
        print(f"[Early Stopping] No improvement for {ckpt.patience} epochs. Halting at Epoch {epoch}.")
        print(f"Best Model: Epoch {ckpt.best_epoch} with Val Loss: {ckpt.best_loss:.4f}")
        print("=" * 95)
        break

print("\n" + "=" * 95)
print(f"Training Finished! Best Val Loss: {ckpt.best_loss:.4f} achieved at Epoch {ckpt.best_epoch}.")
print("=" * 95)


In [ ]:
# CELL 19 - Load Best Checkpoint & Compute Validation mAP
logger.info("=" * 70)
logger.info("[CELL 19] Loading best checkpoint and evaluating validation mAP...")
logger.info("=" * 70)

ckpt.load_best()
val_map_results = compute_map(lwdetr, val_loader, device, img_size=RESOLUTION, threshold=MAP_EVAL_THRESHOLD)

logger.info("=" * 50)
logger.info("Validation mAP Metrics:")
for k, v in val_map_results.items():
    if isinstance(v, torch.Tensor):
        if v.numel() == 1:
            logger.info(f"   - {k:20s}: {v.item():.4f}")
        else:
            logger.info(f"   - {k:20s}: {[round(x, 4) for x in v.tolist()]}")
    else:
        logger.info(f"   - {k:20s}: {v}")
logger.info("=" * 50)


In [ ]:
# CELL 20 - Export Best Model to model/ Directory
logger.info("=" * 70)
logger.info("[CELL 20] Exporting best model checkpoint...")
logger.info("=" * 70)

exported_path = os.path.join(FINAL_MODEL_DIR, f"best_model_{MODE_TAG}.pth")
if os.path.exists(ckpt.best_path):
    shutil.copy2(ckpt.best_path, exported_path)
    logger.info(f"   - Canonical best model saved to: {exported_path} ({os.path.getsize(exported_path) / 1e6:.1f} MB)")
else:
    logger.warning(f"   Checkpoint not found at {ckpt.best_path}")

logger.info("Export completed.")

In [ ]:
# CELL 21 - Test Set Inference & Visual Predictions (Pure PIL & IPython - Zero Matplotlib Dependency)
from PIL import Image
from IPython.display import Image as IPImage, display

logger.info("=" * 70)
logger.info(f"[CELL 21] Running inference on test split (Confidence={CONFIDENCE}, NMS={NMS_THRESHOLD})...")
logger.info("=" * 70)

import torch.multiprocessing as mp
try:
    mp.set_sharing_strategy('file_system')
except Exception:
    pass
try:
    import resource
    rlimit = resource.getrlimit(resource.RLIMIT_NOFILE)
    resource.setrlimit(resource.RLIMIT_NOFILE, (max(rlimit[0], 4096), max(rlimit[1], 4096)))
except Exception:
    pass

test_ds = COCODetectionDataset(str(IMAGES_DIR), TEST_ANN, RESOLUTION)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=0, collate_fn=collate_fn)

test_map_results = compute_map(lwdetr, test_loader, device, img_size=RESOLUTION, threshold=CONFIDENCE_THRESHOLD)
logger.info("Test Set mAP Results:")
for k, v in test_map_results.items():
    if isinstance(v, torch.Tensor):
        if v.numel() == 1:
            logger.info(f"   - {k:20s}: {v.item():.4f}")
        else:
            logger.info(f"   - {k:20s}: {[round(x, 4) for x in v.tolist()]}")

# Visual sample test predictions using supervision + PIL save/display
color_palette = sv.ColorPalette.from_hex([
    "#e6194B", "#3cb44b", "#ffe119", "#4363d8", "#f58231",
    "#911eb4", "#42d4f4", "#f032e6", "#bfef45", "#fabed4", "#469990"
])
box_annotator = sv.BoxAnnotator(color=color_palette, thickness=2)
label_annotator = sv.LabelAnnotator(color=color_palette, text_scale=0.5, text_thickness=1)

os.makedirs(INFERENCE_OUTPUT_DIR, exist_ok=True)
lwdetr.eval()
preview_count = 0

with torch.no_grad():
    for images, targets in test_loader:
        if preview_count >= NUM_INFERENCE_PREVIEWS:
            break
        images = images.to(device)
        with torch.amp.autocast(device_type="cuda", enabled=torch.cuda.is_available()):
            outputs = lwdetr(images)
        pred_logits = outputs["pred_logits"][0]
        pred_boxes = outputs["pred_boxes"][0]

        scores, class_ids = pred_logits.sigmoid().max(-1)
        keep = scores > CONFIDENCE
        if keep.sum() == 0:
            continue

        boxes_xyxy = (box_cxcywh_to_xyxy(pred_boxes[keep]) * RESOLUTION).cpu().numpy()
        scores_np = scores[keep].cpu().numpy()
        class_ids_np = class_ids[keep].cpu().numpy()

        img_np = (images[0].permute(1, 2, 0).cpu().numpy() * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406]))
        img_np = np.clip(img_np * 255, 0, 255).astype(np.uint8)

        detections = sv.Detections(xyxy=boxes_xyxy, confidence=scores_np, class_id=class_ids_np)
        labels = [f"{class_names[cid] if cid < len(class_names) else str(cid)} {c:.2f}" for cid, c in zip(class_ids_np, scores_np)]
        annotated = box_annotator.annotate(scene=img_np.copy(), detections=detections)
        annotated = label_annotator.annotate(scene=annotated, detections=detections, labels=labels)

        # Save annotated image directly and display via IPython (no Matplotlib)
        out_img_path = os.path.join(INFERENCE_OUTPUT_DIR, f"preview_detection_{preview_count+1}.png")
        Image.fromarray(annotated).save(out_img_path)
        logger.info(f"Preview #{preview_count+1}: {len(boxes_xyxy)} objects detected -> Saved to {out_img_path}")
        display(IPImage(filename=out_img_path))
        preview_count += 1

logger.info(f"Successfully generated and previewed {preview_count} test detection images.")


In [ ]:
# CELL 22 - Save Test Predictions JSON
logger.info("=" * 70)
logger.info("[CELL 22] Saving test predictions...")
logger.info("=" * 70)

pred_out_file = os.path.join(INFERENCE_OUTPUT_DIR, f"test_predictions_{MODE_TAG}.json")
lwdetr.eval()
all_test_predictions = {}

with torch.no_grad():
    for images, targets in test_loader:
        img_id = targets[0]["image_id"].item()
        orig_h, orig_w = targets[0]["orig_size"].tolist()
        images = images.to(device)
        with torch.amp.autocast(device_type="cuda", enabled=torch.cuda.is_available()):
            outputs = lwdetr(images)
        pred_logits = outputs["pred_logits"][0]
        pred_boxes = outputs["pred_boxes"][0]

        scores, class_ids = pred_logits.sigmoid().max(-1)
        keep = scores > CONFIDENCE

        boxes_xyxy = (box_cxcywh_to_xyxy(pred_boxes[keep]).cpu().numpy() * np.array([orig_w, orig_h, orig_w, orig_h])).tolist()
        scores_list = scores[keep].cpu().tolist()
        class_ids_list = class_ids[keep].cpu().tolist()

        all_test_predictions[str(img_id)] = {
            "boxes_xyxy": boxes_xyxy,
            "confidence": scores_list,
            "class_id": class_ids_list,
            "orig_size": [orig_h, orig_w]
        }

with open(pred_out_file, "w", encoding="utf-8") as f:
    json.dump({
        "pipeline": PIPELINE_NAME,
        "mode": MODE_TAG,
        "num_classes": NUM_CLASSES,
        "class_names": class_names,
        "confidence_threshold": CONFIDENCE,
        "predictions": all_test_predictions
    }, f, indent=2)

logger.info(f"Saved test predictions to: {pred_out_file}")

In [ ]:
# CELL 23 - Final Execution Summary
logger.info("=" * 70)
logger.info(f"[CELL 23] Execution Summary: {PIPELINE_NAME}")
logger.info("=" * 70)
logger.info(f"   - Pipeline Mode:        {MODE_TAG.upper()}")
logger.info(f"   - Number of Classes:    {NUM_CLASSES} ({class_names})")
logger.info(f"   - Architecture:         RF-DETR-{MODEL_SIZE.capitalize()} (resolution={RESOLUTION})")
logger.info(f"   - Effective Batch Size: {BATCH_SIZE * GRAD_ACCUMULATION} (Batch={BATCH_SIZE}, Accum={GRAD_ACCUMULATION})")
logger.info(f"   - Learning Rate:        {LEARNING_RATE} (Weight Decay: {WEIGHT_DECAY})")
logger.info(f"   - Dataset Folder:       {DATASET_DIR}")
logger.info(f"   - Images Folder:        {IMAGES_DIR} (Zero-copy shared repository)")
logger.info(f"   - Checkpoint Folder:    {CHECKPOINT_DIR}")
logger.info(f"   - Exported Model:       {exported_path}")
logger.info(f"   - Test Predictions:     {pred_out_file}")
logger.info(f"   - Pipeline Log File:    {log_file_path}")
logger.info("=" * 70)
logger.info("Multi-Class Pipeline execution completed successfully!")